# Numerical checks

This notebook checks the implementation, and every quantitative statement in the paper that
cannot be read directly off a figure. Each section quotes the statement it checks, and every
check ends in an assertion, so `make checks` (which executes this notebook) fails loudly if
any result changes. Sections 1-3 and 5 run from scratch (section 3 compiles a C kernel); the
others load saved results from `data/generated/`. Checks printed alongside a figure (the
Theorem 1 margins of the Fig. S3 examples, the Fig. S8 endpoints) are in `sup_figs.ipynb`.

## 1. Payoff gradients and the learning step (Secs. II A and II B)

The closed-form pairwise gradient in `WellMixedReactive` is the derivative of the payoff $\pi$ of Sec. II A (checked by finite differences), with and without self-play; the vectorized, low-memory and kernel-matrix (used for Fig. S9) implementations agree; one learning step is the projected gradient step; and a population at GTFT is a fixed point of the full dynamics (Sec. III D).

In [1]:
import numpy as np

from reactive_learning.agents import WellMixedReactive
from reactive_learning.perturbation import gtft_strategy

B, C, EPSILON = 1.0, 0.5, 1e-3


def payoff(x1, x2):
    """Long-run payoff to x1 = (p1, q1) against x2 (Sec. II A)."""
    (p1, q1), (p2, q2) = x1, x2
    r1, r2 = p1 - q1, p2 - q2
    return (B * (q1 * r2 + q2) - C * (q2 * r1 + q1)) / (1 - r1 * r2)


rng = np.random.default_rng(0)
agents = rng.uniform(EPSILON, 1 - EPSILON, (60, 2))
eta = rng.uniform(0.01, 0.39, len(agents))
h = 1e-6
# Central differences in the focal agent's own strategy; opponents (itself included) fixed.
pairwise = np.array([
    [[(payoff(x + h * e, y) - payoff(x - h * e, y)) / (2 * h) for e in np.eye(2)] for y in agents]
    for x in agents
])  # (focal, opponent, p/q)


def model(**kwargs):
    return WellMixedReactive(agents.copy(), eta, b=B, c=C, epsilon=EPSILON, **kwargs)


vectorized = model().compute_global_gradients()
low_memory = model(vectorized_threshold=0).compute_global_gradients()
no_self = model(self_interactions=False).compute_global_gradients()
matmul = model(matmul_gradients=True).compute_global_gradients()  # basin sweep (Fig. S9)
matmul_no_self = model(matmul_gradients=True, self_interactions=False).compute_global_gradients()
off_diagonal = (pairwise.sum(axis=1) - pairwise[np.arange(len(agents)), np.arange(len(agents))])
error = np.max(np.abs(vectorized - pairwise.mean(axis=1)))
print(f"closed-form vs finite-difference gradient (with self-play): {error:.1e}")
print(f"vectorized vs low-memory gradient: {np.max(np.abs(vectorized - low_memory)):.1e}")
print(f"without self-play vs finite differences over j != i: "
      f"{np.max(np.abs(no_self - off_diagonal / (len(agents) - 1))):.1e}")
assert error < 1e-8
print(f"kernel-matrix (matmul) vs vectorized gradient: {np.max(np.abs(matmul - vectorized)):.1e}")
assert np.allclose(vectorized, low_memory, rtol=0, atol=1e-13)
assert np.allclose(matmul, vectorized, rtol=0, atol=1e-13)
assert np.allclose(matmul_no_self, no_self, rtol=0, atol=1e-13)
assert np.allclose(no_self, off_diagonal / (len(agents) - 1), rtol=0, atol=1e-8)

# One learning step is the projected (clipped) gradient step of Sec. II B.
system = model()
system.step()
expected = np.clip(agents + eta[:, None] * vectorized, EPSILON, 1 - EPSILON)
assert np.array_equal(system.agents, expected)

# Sec. III D: a population concentrated at GTFT is a fixed point of the full dynamics.
gtft = np.tile(gtft_strategy(b=B, c=C, epsilon=EPSILON), (10, 1))
gtft_system = WellMixedReactive(gtft, np.full(10, 0.2), b=B, c=C, epsilon=EPSILON)
residual = np.max(np.abs(gtft_system.compute_global_gradients()))
print(f"largest gradient component at GTFT: {residual:.1e}")
assert residual < 1e-15

closed-form vs finite-difference gradient (with self-play): 2.0e-11
vectorized vs low-memory gradient: 3.9e-16
without self-play vs finite differences over j != i: 2.0e-11
kernel-matrix (matmul) vs vectorized gradient: 5.6e-16
largest gradient component at GTFT: 0.0e+00


## 2. Infinite-population lattice model (Supplement S1.3)

Each lattice point moves with the same gradient as an agent at that point, and the bilinear redistribution conserves the mass $1/L$ of each learning-rate grid.

In [2]:
import numpy as np

from reactive_learning.agents import WellMixedReactive
from reactive_learning.lattice import ReactiveLattice

EPSILON, N_POINTS = 1e-3, 12
eta = np.linspace(0.01, 0.39, 4)

# Equal mass on every point: the lattice's gradient must be the agent model's gradient
# for one agent per lattice point.
lattice = ReactiveLattice(np.full((N_POINTS, N_POINTS, eta.size), 1 / (N_POINTS**2 * eta.size)),
                          eta=eta, epsilon=EPSILON)
points = lattice.lattice.reshape(-1, 2)
gradients = WellMixedReactive(points, np.ones(len(points)), b=1.0, c=0.5,
                              epsilon=EPSILON).compute_global_gradients()
expected = np.clip(points[:, None, :] + eta[None, :, None] * gradients[:, None, :],
                   EPSILON, 1 - EPSILON)
error = np.max(np.abs(lattice.new_locations().reshape(-1, eta.size, 2) - expected))
print(f"lattice vs agent-model update: {error:.1e}")
assert error < 1e-14

# Bilinear redistribution conserves each learning rate's mass (1/L) and keeps it nonnegative.
rng = np.random.default_rng(1)
freq = rng.random((N_POINTS, N_POINTS, eta.size))
lattice = ReactiveLattice(freq / (freq.sum(axis=(0, 1)) * eta.size), eta=eta, epsilon=EPSILON)
for _ in range(50):
    lattice.step()
mass_error = np.max(np.abs(lattice.freq.sum(axis=(0, 1)) - 1 / eta.size))
print(f"mass per learning rate after 50 steps: deviation {mass_error:.1e}")
assert mass_error < 1e-13 and np.all(lattice.freq >= 0)

lattice vs agent-model update: 1.1e-16
mass per learning rate after 50 steps: deviation 1.9e-16


## 3. Compiled kernel (Supplement S5)

> The calculation was implemented in compiled C ... the implementation was checked against direct pairwise calculations.

The C kernel used by the long run, with direct sums and with the moment expansion, reproduces 100 NumPy updates. Requires a C compiler.

In [3]:
import tempfile
from pathlib import Path

import numpy as np

from reactive_learning.agents import WellMixedReactive
from reactive_learning.long_run import advance, compile_kernel

rng = np.random.default_rng(42)
worst = 0.0
with tempfile.TemporaryDirectory() as directory:
    lib = compile_kernel(Path(directory))
    for epsilon in (1e-2, 1e-3):
        # A spread-out population tests the direct sums; a narrow cluster on
        # p = 1 - epsilon tests the moment expansion.
        for initial in (rng.uniform(epsilon, 1 - epsilon, (40, 2)),
                        np.column_stack([np.full(40, 1 - epsilon), rng.uniform(0.3, 0.4, 40)])):
            eta = rng.uniform(0.01, 0.39, 40)
            reference = WellMixedReactive(initial.copy(), eta, b=1, c=0.5, epsilon=epsilon)
            reference.run(100)
            for tolerance in (0.0, 1e-14):  # 0 selects direct pairwise sums
                actual = initial.copy()
                advance(lib, actual, eta, np.full(40, 1 / 40), 100, 1, 0.5, epsilon, tolerance)
                worst = max(worst, np.max(np.abs(actual - reference.agents)))
print(f"largest difference from NumPy after 100 updates: {worst:.1e}")
assert worst < 1e-11

largest difference from NumPy after 100 updates: 7.8e-16


## 4. Self-play (Sec. II B)

> Note that we include self-play, though this does not meaningfully affect results.

Both main populations rerun without self-play (`scripts/generate/self_interactions_data.py`) still end at ALLD (uniform rates) and in the GTFT regime (variable rates). The GTFT regime means at least 99% of agents on $p = 1-\varepsilon$ with $q > \varepsilon$ and cooperation of at least 95%, at every check over 5,000 updates. ALLD is observed or certified by Theorem 1. These are finite-time checks, not proofs of convergence.

In [4]:
import numpy as np

from reactive_learning import robustness
from reactive_learning.paths import DATA_DIR


def verify_no_self_case(regime):
    epsilon, self_interactions = 1e-3, False
    label = f"{regime:8s}, epsilon={epsilon:g}, self_interactions={self_interactions}"
    path = DATA_DIR / "self_interactions" / f"{regime}_eps{epsilon:g}_self{int(self_interactions)}.npz"
    if not path.exists():
        print(f"MISSING: {label} — run scripts/generate/self_interactions_data.py")
        return False
    with np.load(path) as data:
        assert float(data["epsilon"]) == epsilon
        assert bool(data["self_interactions"]) == self_interactions
        assert str(data["regime"]) == regime
        assert int(data["n"]) == 2500 and int(data["seed"]) == 29
        assert float(data["b"]) == 1 and float(data["c"]) == 0.5
        assert float(data["noise_sd"]) == 0
        assert int(data["hold_steps"]) >= 5000
        assert int(data["check_every"]) <= 1000
        steps, states = data["steps"], data["agents"]
        rng = np.random.RandomState(29)
        expected_initial = np.clip(rng.random((2500, 2)), epsilon, 1 - epsilon)
        variable_eta = 0.01 + 0.38 * rng.random(2500)
        expected_eta = variable_eta if regime == "variable" else np.full(2500, 0.2)
        np.testing.assert_array_equal(states[0], expected_initial)
        np.testing.assert_array_equal(data["eta"], expected_eta)
        assert np.all(np.diff(steps) > 0)
        assert np.all(np.isfinite(states))
        assert np.all((states >= epsilon) & (states <= 1 - epsilon))
        parameters = dict(b=1, c=0.5, epsilon=epsilon,
                          self_interactions=self_interactions)
        final = robustness.endpoint_evidence(states[-1], **parameters)
        alld = final["alld_observed"] or final["alld_margin"] > 1e-8
        start = int(data["gtft_since"])
        held = start >= 0 and int(steps[-1]) - start >= 5000
        if held:
            held = start in steps and all(
                robustness.endpoint_evidence(state, **parameters)["gtft_regime"]
                for state in states[steps >= start]
            )
        verified = alld if regime == "uniform" else held
        contrary = held if regime == "uniform" else (alld or final["cooperation_excluded"])
        verdict = "PASS" if verified else ("FAIL" if contrary else "UNRESOLVED")
        if alld:
            detail = ("ALLD observed" if final["alld_observed"] else
                      f"ALLD certified, margin={final['alld_margin']:.3e}")
        elif final["cooperation_excluded"]:
            detail = (f"permanent ALLD cohort={final['permanent_alld_fraction']:.2%}; "
                      f"cooperation can never exceed {final['cooperation_ceiling']:.2%}; "
                      "remaining agents need not converge to ALLD")
        else:
            detail = (f"ALLD fraction={final['alld_fraction']:.2%}, "
                      f"GTFT fraction={final['gtft_fraction']:.2%}, "
                      f"cooperation={final['cooperation']:.4f}, "
                      f"held for {int(steps[-1]) - start if start >= 0 else 0:,} updates")
        status = robustness.stopping_status(
            final, step=int(steps[-1]), gtft_since=start,
            hold_steps=int(data["hold_steps"]),
        )
        if status != "unresolved":
            completion = "stopping condition reached"
        elif steps[-1] >= int(data["requested_steps"]):
            completion = "step limit reached; run complete but outcome unresolved"
        else:
            completion = "saved checkpoint; run unfinished"
        print(f"{verdict}: {label} — {detail} (step {steps[-1]:,}; {completion})")
        return verified


print("=== NO SELF-INTERACTIONS: epsilon=1e-3 ===")
passed = sum(verify_no_self_case(regime) for regime in ("uniform", "variable"))
print(f"\nVerified {passed}/2 expected results without self-interactions.")
assert passed == 2

=== NO SELF-INTERACTIONS: epsilon=1e-3 ===
PASS: uniform , epsilon=0.001, self_interactions=False — ALLD certified, margin=4.337e-02 (step 2,000; stopping condition reached)
PASS: variable, epsilon=0.001, self_interactions=False — ALLD fraction=0.00%, GTFT fraction=99.88%, cooperation=0.9978, held for 5,000 updates (step 15,000; stopping condition reached)

Verified 2/2 expected results without self-interactions.


## 5. Gradients during the collapse of forgiveness (Sec. III A, Supplement S2)

Closed forms of the gradient against the uniform initial distribution, checked against quadrature; its largest component ($-0.157536$, at TFT) and range of directions ($26.67^\circ$ to $63.33^\circ$ left of the downward vertical); the constant field of a population at ALLD and the bound $\varepsilon(1 + \eta c)$ on the loss of reciprocity; and the gradients of the seed-29 initial population.

In [5]:
import numpy as np
from scipy import integrate

from reactive_learning.agents import WellMixedReactive

B, C, EPSILON = 1.0, 0.5, 1e-3
SERIES_CUTOFF = 0.3          # |wL| below which the series branch is used
TERMS = 40


def moments(w, L):
    """J_0, J_1, J_2 of Supplement eq. for the uniform initial distribution."""
    zeta = w * L
    if abs(zeta) < SERIES_CUTOFF:
        # J_k = (series in zeta) / w^{k+2}; the k=0,1 leading terms are exact at w=0.
        j = np.arange(1, TERMS)
        J0 = L**2 * np.sum(zeta ** (2 * j - 2) / j)
        j = np.arange(2, TERMS)
        J1 = 2 * np.sum(L ** (2 * j) * w ** (2 * j - 3) * (j - 1) / (j * (2 * j - 1)))
        J2 = np.sum(L ** (2 * j) * w ** (2 * j - 4) * (2 * j - 3) / (j * (2 * j - 1)))
        return J0, J1, J2
    lp, lm = np.log1p(zeta), np.log1p(-zeta)
    return (
        -(lm + lp) / w**2,
        (-(zeta + 2) * lp + (zeta - 2) * lm) / w**3,
        (zeta**2 - (2 * zeta + 3) * lp + (2 * zeta - 3) * lm) / w**4,
    )


def uniform_gradient(p1, q1, b=B, c=C, epsilon=EPSILON):
    """Gradient against the uniform distribution on X, via Phi_w."""
    L = 1 - 2 * epsilon
    J0, J1, J2 = moments(p1 - q1, L)
    phi = lambda B_: (B_ * (b * J2 - c * J1) + (b * J1 - c * J0) / 2) / L**2
    return phi(q1 - 0.5), phi(0.5 - p1)


def quad_moment(k, w, L):
    return integrate.quad(lambda r: r**k * (L - abs(r)) / (1 - w * r) ** 2, -L, L,
                          points=[0.0], epsabs=1e-12, epsrel=1e-12)[0]


def quad_gradient(p1, q1, b=B, c=C, epsilon=EPSILON):
    """Gradient against the uniform distribution by direct 2-D quadrature."""
    L, r1 = 1 - 2 * epsilon, p1 - q1
    lo, hi = epsilon, 1 - epsilon
    dp = lambda q2, p2: (b * (p2 - q2) - c) * (q1 * (p2 - q2) + q2) / (1 - r1 * (p2 - q2)) ** 2
    dq = lambda q2, p2: (b * (p2 - q2) - c) * (1 - q2 - p1 * (p2 - q2)) / (1 - r1 * (p2 - q2)) ** 2
    return tuple(
        integrate.dblquad(f, lo, hi, lo, hi, epsabs=1e-13, epsrel=1e-13)[0] / L**2
        for f in (dp, dq)
    )


print("=== MOMENTS AGAINST QUADRATURE (both branches) ===")
L = 1 - 2 * EPSILON
worst = {"series": 0.0, "log": 0.0}
for w in [0.0, 1e-6, 1e-3, 0.1, 0.2, 0.31, 0.5, 0.9, L, -1e-3, -0.2, -0.5, -L]:
    branch = "series" if abs(w * L) < SERIES_CUTOFF else "log"
    closed = moments(w, L)
    for k in range(3):
        reference = quad_moment(k, w, L)
        error = abs(closed[k] - reference) / max(1.0, abs(reference))
        worst[branch] = max(worst[branch], error)
for branch, error in worst.items():
    print(f"  {branch:6s} branch: max relative error {error:.2e}")
    assert error < 1e-9, branch

print("\n=== Phi IDENTITY AGAINST 2-D QUADRATURE ===")
rng = np.random.default_rng(29)
worst_identity = 0.0
for _ in range(12):
    p1, q1 = rng.uniform(EPSILON, 1 - EPSILON, 2)
    closed, reference = uniform_gradient(p1, q1), quad_gradient(p1, q1)
    worst_identity = max(worst_identity, *(abs(a - b) for a, b in zip(closed, reference)))
print(f"  max absolute discrepancy: {worst_identity:.2e}")
assert worst_identity < 1e-10

print("\n=== EXTREMA LIE ON THE BOUNDARY OF X ===")
grid = np.linspace(EPSILON, 1 - EPSILON, 160)
interior = max(max(uniform_gradient(p, q)) for p in grid[1:-1] for q in grid[1:-1])
boundary = max(
    max(uniform_gradient(p, q))
    for p, q in ([(x, EPSILON) for x in grid] + [(x, 1 - EPSILON) for x in grid]
                 + [(EPSILON, y) for y in grid] + [(1 - EPSILON, y) for y in grid])
)
print(f"  interior maximum {interior:+.6f}  <=  boundary maximum {boundary:+.6f}")
assert interior <= boundary + 1e-12

def boundary_max(epsilon, cb, n=1400):
    eps_grid = np.linspace(epsilon, 1 - epsilon, n)
    edges = ([(x, epsilon) for x in eps_grid] + [(x, 1 - epsilon) for x in eps_grid]
             + [(epsilon, y) for y in eps_grid] + [(1 - epsilon, y) for y in eps_grid])
    return max(max(uniform_gradient(p, q, b=1.0, c=cb, epsilon=epsilon)) for p, q in edges)


largest = boundary_max(EPSILON, C / B)
print(f"\n=== LARGEST COMPONENT ===\n  largest component at b/c={B / C:g}: {largest:+.6f} (Supplement S2.1: -0.157536)")
assert abs(largest + 0.157536) < 1e-5

tft_value = max(uniform_gradient(1 - EPSILON, EPSILON))
print(f"  largest component at TFT = (1-eps, eps): {tft_value:+.6f}")
assert abs(tft_value - largest) < 1e-12

print("\n=== DIRECTIONS OF THE INITIAL GRADIENT (Supplement S2.1: 26.67 to 63.33 deg) ===")
fine = np.linspace(EPSILON, 1 - EPSILON, 400)
angles = np.degrees([np.arctan2(-gp, -gq) for gp, gq in
                     (uniform_gradient(p, q) for p in fine for q in fine)])
print(f"  angle left of the downward vertical: {angles.min():.2f} to {angles.max():.2f} deg")
assert round(angles.min(), 2) == 26.67 and round(angles.max(), 2) == 63.33

print("=== THE ALLD FIELD IS CONSTANT ===")
worst_field = 0.0
for _ in range(400):
    p1, q1 = rng.uniform(EPSILON, 1 - EPSILON, 2)
    r1, r2, q2 = p1 - q1, 0.0, EPSILON
    field = ((B * r2 - C) * (q1 * r2 + q2) / (1 - r1 * r2) ** 2,
             (B * r2 - C) * (1 - q2 - p1 * r2) / (1 - r1 * r2) ** 2)
    worst_field = max(worst_field, abs(field[0] + C * EPSILON),
                      abs(field[1] + C * (1 - EPSILON)))
print(f"  max deviation from (-c eps, -c(1-eps)): {worst_field:.2e}")
assert worst_field == 0.0
print(f"  angle from vertical: {np.degrees(np.arctan(EPSILON / (1 - EPSILON))):.4f} deg; "
      f"forgiveness falls {(1 - EPSILON) / EPSILON:.0f} times as fast")

print("\n=== THE O(eps) RECIPROCITY BOUND ===")
worst_slack = -np.inf
for _ in range(600):
    p, q = rng.uniform(EPSILON, 1 - EPSILON, 2)
    p0, q0 = p, q
    eta = rng.uniform(1e-3, 0.5)
    while q > EPSILON:      # constant field, so the trajectory is explicit
        p = max(EPSILON, p - eta * C * EPSILON)
        q = max(EPSILON, q - eta * C * (1 - EPSILON))
    bound = EPSILON / (1 - EPSILON) * (q0 - EPSILON) + eta * C * EPSILON
    worst_slack = max(worst_slack, (p0 - p) - bound)
    assert bound < EPSILON * (1 + eta * C)
print(f"  max (loss - bound) over 600 starts and rates: {worst_slack:.2e}  (must be <= 0)")
assert worst_slack <= 1e-12

print("\n=== THE SAMPLED INITIAL POPULATION ===")
initial = np.clip(np.random.RandomState(29).random((2500, 2)), EPSILON, 1 - EPSILON)
system = WellMixedReactive(initial.copy(), np.full(len(initial), 0.2),
                           b=B, c=C, epsilon=EPSILON, vectorized_threshold=0)
gradients = system.compute_global_gradients()
print(f"  agents with a negative p gradient: {np.mean(gradients[:, 0] < 0):.2%}")
print(f"  agents with a negative q gradient: {np.mean(gradients[:, 1] < 0):.2%}")
print(f"  largest component over the population: {gradients.max():+.6f}")
assert np.all(gradients < 0)
print("  every agent of the seed-29 initial population moves down and to the left.")

=== MOMENTS AGAINST QUADRATURE (both branches) ===
  series branch: max relative error 2.18e-16
  log    branch: max relative error 6.00e-15

=== Phi IDENTITY AGAINST 2-D QUADRATURE ===
  max absolute discrepancy: 3.50e-15

=== EXTREMA LIE ON THE BOUNDARY OF X ===
  interior maximum -0.163951  <=  boundary maximum -0.157536



=== LARGEST COMPONENT ===
  largest component at b/c=2: -0.157536 (Supplement S2.1: -0.157536)
  largest component at TFT = (1-eps, eps): -0.157536

=== DIRECTIONS OF THE INITIAL GRADIENT (Supplement S2.1: 26.67 to 63.33 deg) ===


  angle left of the downward vertical: 26.67 to 63.33 deg
=== THE ALLD FIELD IS CONSTANT ===
  max deviation from (-c eps, -c(1-eps)): 0.00e+00
  angle from vertical: 0.0574 deg; forgiveness falls 999 times as fast

=== THE O(eps) RECIPROCITY BOUND ===
  max (loss - bound) over 600 starts and rates: -5.36e-08  (must be <= 0)

=== THE SAMPLED INITIAL POPULATION ===
  agents with a negative p gradient: 100.00%
  agents with a negative q gradient: 100.00%
  largest component over the population: -0.162814
  every agent of the seed-29 initial population moves down and to the left.


## 6. Entering the transient unforgiving state (Secs. III A and III B, Fig. S3)

> Although the variable learning rate population has more agents playing the ALLD strategy, it also has support on higher values of $p$.

> Uniform learning rates produce a distribution with a higher mean, but lower maximum. (Fig. S3 caption)

In [6]:
import numpy as np

from reactive_learning.axis_entry import whole_population_on_axis
from reactive_learning.paths import DATA_DIR

with np.load(DATA_DIR / "axis_entry.npz") as data:
    conditions, entry_steps, entry_agents = data["conditions"], data["entry_steps"], data["agents"]
    epsilon = float(data["epsilon"])

stats = {}
for condition, step, agents in zip(conditions, entry_steps, entry_agents, strict=True):
    assert whole_population_on_axis(agents, epsilon=epsilon, atol=0)
    p = agents[:, 0]
    stats[str(condition)] = alld, mean, top = np.mean(p == epsilon), p.mean(), p.max()
    print(f"{condition}: first step with every q = epsilon is {step}; "
          f"{alld:.2%} of agents exactly at ALLD, mean p = {mean:.3f}, max p = {top:.4f}")

uniform, variable = stats["Uniform learning rates"], stats["Variable learning rates"]
assert variable[0] > uniform[0]  # "the variable learning rate population has more ... ALLD"
assert uniform[1] > variable[1]  # "Uniform learning rates produce a distribution with a higher mean,
assert uniform[2] < variable[2]  #  but lower maximum" (Fig. S3 caption)

Uniform learning rates: first step with every q = epsilon is 18; 32.56% of agents exactly at ALLD, mean p = 0.268, max p = 0.8116
Variable learning rates: first step with every q = epsilon is 270; 53.44% of agents exactly at ALLD, mean p = 0.167, max p = 0.9990


## 7. Theorem 1 along the unforgiving line (Sec. III B, Supplement S3)

> By step 1900, the uniform learning rate population has $U < 0$ ... The variable learning rates population evolves towards TFT and then departs the $q = \varepsilon$ axis before having $P > Q$.

Also Step 0 of the proof: on the line, every agent whose $p$-gradient is negative also has a negative $q$-gradient.

In [7]:
import numpy as np

from reactive_learning.agents import WellMixedReactive
from reactive_learning.axis_entry import whole_population_on_axis
from reactive_learning.paths import DATA_DIR
from reactive_learning.transient_unforgiving import empirical_convergence_bounds

for regime in ("uniform", "variable"):
    with np.load(DATA_DIR / f"agent_{regime}.npz") as data:
        steps, agents, eta = data["steps"], data["agents"], data["eta"]
        b, c, epsilon = (float(data[key]) for key in ("b", "c", "epsilon"))
    window = np.flatnonzero([whole_population_on_axis(state, epsilon=epsilon) for state in agents])
    assert np.array_equal(window, np.arange(window[0], window[-1] + 1)), "left and re-entered"
    bounds = [empirical_convergence_bounds(agents[i], b=b, c=c, epsilon=epsilon) for i in window]
    alld = np.array([bound.alld_margin for bound in bounds])  # -U
    tft = np.array([bound.tft_margin for bound in bounds])  # P - Q
    on_line = steps[window]

    # Supplement S3, Step 0: on the line a falling p forces a falling q, so no agent may
    # have G_p < 0 <= G_q. Check for every agent in every saved snapshot on the line.
    positive_p, violations = [], 0
    for i in window:
        gradients = WellMixedReactive(agents[i], eta, b=b, c=c, epsilon=epsilon,
                                      vectorized_threshold=0).compute_global_gradients()
        positive_p.append(np.mean(gradients[:, 0] > 0))
        violations += np.sum((gradients[:, 0] < 0) & (gradients[:, 1] >= 0))

    print(f"--- {regime} learning rates ---")
    print(f"every q = epsilon in saved steps {on_line[0]:,} to {on_line[-1]:,} "
          f"({window.size} snapshots)" + ("; the population has left the line by step "
          f"{steps[window[-1] + 1]:,}" if window[-1] + 1 < len(steps) else ""))
    print(f"at entry: -U = {alld[0]:+.4f}, P - Q = {tft[0]:+.4f} ({bounds[0].outcome})")
    first_alld = on_line[alld > 0][0] if np.any(alld > 0) else None
    print(f"first saved step with U < 0: {first_alld}; best P - Q on the line: {tft.max():+.4f}")
    print(f"agents with G_p > 0: at most {max(positive_p):.2%} of the population; "
          f"G_p < 0 <= G_q in {violations} agent-snapshots")
    assert violations == 0
    if regime == "uniform":
        assert first_alld == 1900  # "by step 1900, the uniform ... population has U < 0"
    else:
        assert first_alld is None and tft.max() < 0  # "departs the q = eps axis before P > Q"

--- uniform learning rates ---
every q = epsilon in saved steps 18 to 15,000 (232 snapshots)
at entry: -U = -0.4342, P - Q = -0.2266 (undetermined)
first saved step with U < 0: 1900; best P - Q on the line: -0.2266
agents with G_p > 0: at most 0.00% of the population; G_p < 0 <= G_q in 0 agent-snapshots


--- variable learning rates ---
every q = epsilon in saved steps 270 to 2,200 (41 snapshots); the population has left the line by step 2,400
at entry: -U = -31.0128, P - Q = -0.3310 (undetermined)
first saved step with U < 0: None; best P - Q on the line: -0.3310
agents with G_p > 0: at most 1.96% of the population; G_p < 0 <= G_q in 0 agent-snapshots


## 8. Swapping learning rates on entering the line (Sec. III B)

> If we endow the agents with sufficiently low uniform learning rates from this point in time onward, the population still converges to GTFT. Conversely, ... endowing agents with variable learning rates from that point onward, the population reliably ends up at ALLD.

The low uniform rate is $\eta = 0.025$ (`small_rate_long`); with $\eta = 0.2$ the population does not reach GTFT, hence "sufficiently low". Data: `scripts/generate/axis_entry_rate_sensitivity_data.py`.

In [8]:
import numpy as np

from reactive_learning.agents import avg_cooperation
from reactive_learning.paths import DATA_DIR
from reactive_learning.transient_unforgiving import empirical_convergence_bounds

outcomes = {}
for case in ("uniform_to_variable", "switch_to_uniform_0.2", "keep_variable_rates",
             "small_rate_long"):
    with np.load(DATA_DIR / "axis_entry_rate_sensitivity" / f"{case}.npz") as data:
        final, step = data["agents"][-1], int(data["steps"][-1])
        b, c, epsilon = (float(data[key]) for key in ("b", "c", "epsilon"))
        source, source_step = str(data["source_file"]), int(data["source_step"])
        rates, first_boundary = str(data["rate_choice"]), int(data["first_all_p_boundary_step"])
    on_boundary = np.mean(final[:, 0] == 1 - epsilon)
    at_alld = np.mean(np.all(final == epsilon, axis=1))
    outcomes[case] = on_boundary, at_alld
    print(f"{case}: {source} at step {source_step}, rates -> {rates}; after {step:,} updates "
          f"{on_boundary:.2%} on p = 1 - eps, {at_alld:.2%} at ALLD, "
          f"cooperation {avg_cooperation(final):.3f}"
          + (f"; every agent on p = 1 - eps from update {first_boundary:,}"
             if first_boundary >= 0 else ""))
    if case == "uniform_to_variable":
        margin = empirical_convergence_bounds(final, b=b, c=c, epsilon=epsilon, atol=0).alld_margin
        print(f"  Theorem 1 ALLD certificate at the final state: -U = {margin:.4f}")
        assert margin > 0

assert outcomes["keep_variable_rates"][0] > 0.99  # control reproduces the move to p = 1 - eps
assert outcomes["small_rate_long"][0] == 1.0
print(f"\nuniform rate 0.2 instead: {outcomes['switch_to_uniform_0.2'][0]:.2%} on p = 1 - eps "
      "(why the paper says \"sufficiently low\" uniform rates)")

uniform_to_variable: agent_uniform.npz at step 18, rates -> variable; after 2,000 updates 0.00% on p = 1 - eps, 41.12% at ALLD, cooperation 0.001
  Theorem 1 ALLD certificate at the final state: -U = 0.0118
switch_to_uniform_0.2: agent_variable.npz at step 270, rates -> 0.2; after 5,000 updates 0.00% on p = 1 - eps, 90.36% at ALLD, cooperation 0.089
keep_variable_rates: agent_variable.npz at step 270, rates -> original; after 5,000 updates 99.76% on p = 1 - eps, 0.00% at ALLD, cooperation 0.995


small_rate_long: agent_variable.npz at step 270, rates -> 0.025; after 75,000 updates 100.00% on p = 1 - eps, 0.00% at ALLD, cooperation 0.998; every agent on p = 1 - eps from update 22,095

uniform rate 0.2 instead: 0.00% on p = 1 - eps (why the paper says "sufficiently low" uniform rates)


The fact that the variable rate population, when given uniform rate 0.2, does not reach GTFT is likely a numerical quirk rather than something to draw too many conclusions from. The high p learners, in this case, rush straight to TFT (in the original simulation they are slow learners, anchored near their original spots), then move rapidly to ALLC. This large presence at ALLC, while the rest of the population is closer to ALLD (there is a split in gradient direction somewhere along the $q = eps$ line), encourages movement back to ALLD. With a normal variable rate population, the high p learners stay closer to TFT, a better catalyst for cooperation than ALLC. When these agents are given lower uniform rate, their gradient doesn't jump as quickly, giving the rest of the population more time to converge.

## 9. The initial GTFT-like distribution (Sec. III C)

> The initial distribution of $q$ once all agents attain $p = 1-\varepsilon$ at step 5800 has mean 0.41, median 0.40, with standard deviation 0.13.

In [9]:
import numpy as np

from reactive_learning.paths import DATA_DIR

with np.load(DATA_DIR / "agent_variable.npz") as data:
    steps, agents, epsilon = data["steps"], data["agents"], float(data["epsilon"])

first = int(np.flatnonzero(np.all(agents[:, :, 0] == 1 - epsilon, axis=1))[0])
q = agents[first, :, 1]
print(f"first saved step with every agent on p = 1 - eps: {steps[first]:,}; "
      f"q mean {q.mean():.3f}, median {np.median(q):.3f}, SD {q.std():.3f}")
assert steps[first] == 5800
assert (round(q.mean(), 2), round(np.median(q), 2), round(q.std(), 2)) == (0.41, 0.40, 0.13)

first saved step with every agent on p = 1 - eps: 5,800; q mean 0.406, median 0.398, SD 0.133


## 10. Long-run experiment (Supplement S5, Fig. S6)

The reduced population: its sampling from step 25,000 and its state after $10^{10}$ updates. Then, at every saved checkpoint, the moment expansion used by the C kernel against direct pairwise sums.

In [10]:
import numpy as np

from reactive_learning.paths import DATA_DIR

with np.load(DATA_DIR / "agent_variable.npz") as data:
    source, source_step, source_eta = data["agents"][-1], int(data["steps"][-1]), data["eta"]
    epsilon = float(data["epsilon"])
with np.load(DATA_DIR / "agent_variable_long_run_n125_rate100.npz") as data:
    indices, weights, eta = data["source_indices"], data["weights"], data["eta"]
    initial, final = data["agents"][0], data["agents"][-1]
    multiplier = float(data["rate_multiplier"])
    assert int(data["source_step"]) == source_step and int(data["steps"][-1]) == 10**10

off_line = np.flatnonzero(source[:, 0] < 1 - epsilon)
print(f"step {source_step:,}: {len(off_line)} of {len(source)} agents off p = 1 - eps, "
      f"at (p, q) = ({source[off_line[0], 0]:.6f}, {source[off_line[0], 1]:.6f})")
sizes, counts = np.unique(np.round(weights[1:] * len(source)).astype(int), return_counts=True)
print("strata of the remaining agents: "
      + ", ".join(f"{n} of size {size}" for size, n in zip(sizes, counts, strict=True))
      + f"; retained agent's weight 1/{1 / weights[0]:.0f}; total weight {weights.sum():.15f}")
assert len(off_line) == 1 and indices[0] == off_line[0] and np.isclose(weights[0], 1 / len(source))
assert (round(source[off_line[0], 0], 6), round(source[off_line[0], 1], 3)) == (0.678762, 0.999)
assert dict(zip(sizes, counts, strict=True)) == {20: 105, 21: 19}
assert np.array_equal(initial, source[indices])
assert np.allclose(eta, multiplier * source_eta[indices])

p, q = final.T
print(f"after 10^10 updates: retained agent at (p, q) = ({p[0]:.6f}, {q[0]:.6f}); others have "
      f"p in [{p[1:].min():.6f}, {p[1:].max():.6f}], q in [{q[1:].min():.6f}, {q[1:].max():.6f}]")
assert (round(p[0], 6), round(q[0], 6)) == (0.998994, 0.50209)  # (0.998994, 0.502090)
assert np.all(p[1:] == 1 - epsilon) and abs(q[1:].mean() - 0.499) < 1e-3

step 25,000: 1 of 2500 agents off p = 1 - eps, at (p, q) = (0.678762, 0.999000)
strata of the remaining agents: 105 of size 20, 19 of size 21; retained agent's weight 1/2500; total weight 1.000000000000000
after 10^10 updates: retained agent at (p, q) = (0.998994, 0.502090); others have p in [0.999000, 0.999000], q in [0.498994, 0.498999]


In [11]:
import numpy as np

from reactive_learning.paths import DATA_DIR

TOLERANCE = 1e-14

long_path = DATA_DIR / "agent_variable_long_run_n125_rate100.npz"

with np.load(long_path) as data:
    long_steps = data["steps"]
    long_agents = data["agents"]
    long_weights = data["weights"]
    long_b = float(data["b"])
    long_c = float(data["c"])
    long_special = int(data["special_count"])

if long_special != 1:
    raise ValueError(f"Expected one focal/off-line agent, found {long_special}.")

focal = 0
ordinary = np.arange(len(long_weights)) >= long_special


def exact_gradients(state):
    """Direct O(N^2) weighted payoff gradients."""
    p, q = state.T

    pi, qi = p[:, None], q[:, None]
    pj, qj = p[None, :], q[None, :]

    denom = (1 + pj * qi - qi * qj + pi * (-pj + qj)) ** 2
    common = long_c + long_b * (-pj + qj)

    gp = (-(pj * qi + qj - qi * qj) * common / denom) @ long_weights
    gq = (((qj - 1) + pi * (pj - qj)) * common / denom) @ long_weights

    return np.column_stack((gp, gq))


def moment_gradients(state):
    """Moment-expanded gradients, matching the compiled C kernel."""
    p, q = state.T
    r = p - q
    n = len(r)

    u = long_weights * (long_b * r - long_c)

    isolated = int(np.argmin(r))

    keep = np.arange(n) != isolated
    lo = np.min(r[keep])
    hi = np.max(r[keep])

    center = 0.5 * (hi + lo)
    radius = 0.5 * (hi - lo)

    d = 1.0 - r * center
    inv_d2 = 1.0 / d**2
    z = r * radius / d

    rho = np.max(np.abs(z))
    dmin = min(1.0, np.min(d))
    umax = np.max(np.abs(long_b * r - long_c))

    cap = min(n // 2, 128)

    def truncation_bound(terms):
        if rho >= 1:
            return np.inf
        return (
            2.0 * umax / dmin**2
            * rho**terms
            * ((terms + 1) - terms * rho)
            / (1.0 - rho) ** 2
        )

    terms = 1
    if TOLERANCE > 0 and rho < 1:
        while terms < cap and truncation_bound(terms) > TOLERANCE:
            terms += 1
    else:
        terms = cap

    # Same fallback as the C kernel.
    if terms >= cap:
        return exact_gradients(state), 0.0, terms, rho, False

    mr = np.zeros(terms)
    mq = np.zeros(terms)
    mu = np.zeros(terms)

    for j in range(n):
        if j == isolated:
            continue

        x = (r[j] - center) / radius if radius > 0 else 0.0
        uk = u[j]

        for k in range(terms):
            mr[k] += uk * r[j]
            mq[k] += uk * q[j]
            mu[k] += uk
            uk *= x

    G = np.empty((n, 2))

    r_iso = r[isolated]
    q_iso = q[isolated]
    u_iso = u[isolated]

    for i in range(n):
        sr = sq = su = 0.0

        for k in range(terms - 1, -1, -1):
            sr = sr * z[i] + (k + 1) * mr[k]
            sq = sq * z[i] + (k + 1) * mq[k]
            su = su * z[i] + (k + 1) * mu[k]

        d_iso = 1.0 - r[i] * r_iso
        k_iso = u_iso / d_iso**2

        G[i, 0] = (
            inv_d2[i] * (q[i] * sr + sq)
            + k_iso * (q[i] * r_iso + q_iso)
        )
        G[i, 1] = (
            inv_d2[i] * (su - sq - p[i] * sr)
            + k_iso * (1.0 - q_iso - p[i] * r_iso)
        )

    return G, truncation_bound(terms), terms, rho, True


# ---------------------------------------------------------------------
# Check every saved checkpoint.
# ---------------------------------------------------------------------

max_bound = (-np.inf, None)
max_component_error = (-np.inf, None)
max_focal_relative_error = (-np.inf, None)
max_other_relative_error = (-np.inf, None)

final_results = None
expansion_count = 0
fallback_count = 0

for frame, (step, state) in enumerate(zip(long_steps, long_agents)):
    G_exact = exact_gradients(state)
    G_moment, bound, terms, rho, used_expansion = moment_gradients(state)

    if used_expansion:
        expansion_count += 1
    else:
        fallback_count += 1

    delta = G_moment - G_exact

    exact_norm = np.linalg.norm(G_exact, axis=1)
    error_norm = np.linalg.norm(delta, axis=1)

    relative_error = np.divide(
        error_norm,
        exact_norm,
        out=np.zeros_like(error_norm),
        where=exact_norm > 0,
    )

    component_error = np.max(np.abs(delta))

    if bound > max_bound[0]:
        max_bound = (bound, step)

    if component_error > max_component_error[0]:
        max_component_error = (component_error, step)

    if relative_error[focal] > max_focal_relative_error[0]:
        max_focal_relative_error = (relative_error[focal], step)

    other_rel = np.max(relative_error[ordinary])
    if other_rel > max_other_relative_error[0]:
        max_other_relative_error = (other_rel, step)

    if frame == len(long_steps) - 1:
        final_results = {
            "step": step,
            "terms": terms,
            "rho": rho,
            "bound": bound,
            "focal_gradient": exact_norm[focal],
            "focal_abs_error": error_norm[focal],
            "focal_rel_error": relative_error[focal],
            "min_other_gradient": np.min(exact_norm[ordinary]),
        }


print("=== LONG-RUN NUMERICAL VALIDATION ===")
print(f"Saved checkpoints checked: {len(long_steps)}")
print(f"Moment expansion used:      {expansion_count}")
print(f"Direct-sum fallbacks:       {fallback_count}")

print("\n=== WORST CASE OVER ALL SAVED CHECKPOINTS ===")
print(
    f"Largest rigorous truncation bound per component: "
    f"{max_bound[0]:.3e}  (step {max_bound[1]:,})"
)
print(
    f"Largest observed component discrepancy:          "
    f"{max_component_error[0]:.3e}  "
    f"(step {max_component_error[1]:,})"
)
print(
    f"Largest focal-agent relative gradient error:     "
    f"{max_focal_relative_error[0]:.3e}  "
    f"(step {max_focal_relative_error[1]:,})"
)
print(
    f"Largest other-agent relative gradient error:     "
    f"{max_other_relative_error[0]:.3e}  "
    f"(step {max_other_relative_error[1]:,})"
)

print("\n=== FINAL CHECKPOINT ===")
print(f"Step:                                      {final_results['step']:,}")
print(f"Moment terms retained:                     {final_results['terms']}")
print(f"Expansion parameter rho:                   {final_results['rho']:.3e}")
print(f"Rigorous truncation bound per component:   {final_results['bound']:.3e}")
print(f"Focal gradient magnitude:                  {final_results['focal_gradient']:.3e}")
print(f"Focal absolute gradient discrepancy:       {final_results['focal_abs_error']:.3e}")
print(f"Focal relative gradient discrepancy:       {final_results['focal_rel_error']:.3e}")
print(f"Smallest other-agent gradient magnitude:   {final_results['min_other_gradient']:.3e}")
assert fallback_count == 0 and max_bound[0] <= TOLERANCE

=== LONG-RUN NUMERICAL VALIDATION ===
Saved checkpoints checked: 10149
Moment expansion used:      10149
Direct-sum fallbacks:       0

=== WORST CASE OVER ALL SAVED CHECKPOINTS ===
Largest rigorous truncation bound per component: 9.958e-15  (step 119,000,000)
Largest observed component discrepancy:          1.016e-16  (step 6)
Largest focal-agent relative gradient error:     6.811e-08  (step 9,536,000,000)
Largest other-agent relative gradient error:     1.674e-10  (step 9,821,000,000)

=== FINAL CHECKPOINT ===
Step:                                      10,000,000,000
Moment terms retained:                     3
Expansion parameter rho:                   1.546e-06
Rigorous truncation bound per component:   1.627e-19
Focal gradient magnitude:                  5.550e-14
Focal absolute gradient discrepancy:       1.100e-21
Focal relative gradient discrepancy:       1.981e-08
Smallest other-agent gradient magnitude:   2.793e-11


## 11. Isolated pairs of noisy learners (Discussion, Fig. S11)

> By step 1240, 98.9% of pairs have fallen below 5% cooperation; the remaining 1.1%, with initial cooperation between 0.43 and 0.87, remain near full cooperation. (Fig. S11 caption)

In [12]:
import numpy as np

from reactive_learning.paths import DATA_DIR

with np.load(DATA_DIR / "agent_pairs.npz") as data:
    cooperation, steps = data["average_cooperation"], data["steps"]

initial, final = cooperation[:, 0], cooperation[:, steps.tolist().index(1240)]
defect, cooperate = final < 0.05, final > 0.9
print(f"{len(final)} pairs; at step 1,240 (the last step shown): {defect.mean():.1%} below 5% "
      f"cooperation, {cooperate.mean():.1%} above 90%")
print(f"initial cooperation of the pairs that end above 90%: "
      f"{initial[cooperate].min():.2f} to {initial[cooperate].max():.2f}")
assert round(defect.mean(), 3) == 0.989 and round(cooperate.mean(), 3) == 0.011
assert (round(initial[cooperate].min(), 2), round(initial[cooperate].max(), 2)) == (0.43, 0.87)

1000 pairs; at step 1,240 (the last step shown): 98.9% below 5% cooperation, 1.1% above 90%
initial cooperation of the pairs that end above 90%: 0.43 to 0.87
